# APEX Dublaj — oyuncunun kendi sesiyle Türkçe + dudak senkronu

**Kaggle** (Settings → Accelerator → *GPU T4*, Internet *On*) veya **Colab** (Runtime → *T4 GPU*) üzerinde çalışır.
Hücreleri sırayla çalıştır.

Akış: konuşma/müzik ayrımı (Demucs) → yazıya dökme (Whisper) → konuşmacı ayrımı → Türkçe çeviri →
her oyuncunun **kendi sesiyle** Türkçe seslendirme (XTTS-v2) → **dudak senkronu** (LatentSync) →
müzik + Türkçe konuşma + "Yapay zekâ ile dublajlandı" etiketi.

⚠️ XTTS-v2 yalnızca **ticari olmayan** kullanım içindir (Coqui Public Model License).
Yalnızca haklarına sahip olduğun ve sesi kopyalanan kişilerin izin verdiği videoları kullan;
yayımlarken platformun *yapay zekâ ile değiştirildi* seçeneğini işaretle.

In [ ]:
# ===== AYARLAR =====
VIDEO = ""                 # Doğrudan MP4 bağlantısı veya dosya yolu (Kaggle: /kaggle/input/..., Colab: /content/...)
KAYNAK_DIL = ""            # Ör. "en". Boş = otomatik algıla
DUDAK_SENKRONU = True      # LatentSync (yavaş: T4'te 1 dk video ≈ 5-15 dk)
ALTYAZI_GOM = False        # Türkçe altyazıyı görüntüye göm (kapalıysa ayrı altyazı izi eklenir)
WHISPER_MODEL = "medium"   # small / medium / large-v3

HAKLARIM_VAR_VE_IZIN_ALINDI = False    # Videonun haklarına sahibim, ses sahipleri izin verdi
XTTS_LISANSINI_KABUL_EDIYORUM = False  # https://coqui.ai/cpml — yalnızca ticari olmayan kullanım

# Çeviri: anahtar yoksa yerel NLLB-200 (ücretsiz, kurulumsuz). İsteğe bağlı: ücretsiz Groq anahtarı
# (console.groq.com, kart gerekmez) daha doğal çeviri verir.
CEVIRI_BASE_URL = "https://api.groq.com/openai/v1"
CEVIRI_MODEL = "openai/gpt-oss-120b"
REPO = "https://github.com/FatihKoc81/VIDEO-OTOMASYON"
REPO_BRANCH = "ccr-6840847c-b854ob"   # main'e birleştirildikten sonra "main"

assert VIDEO, "VIDEO ayarını doldur."
assert HAKLARIM_VAR_VE_IZIN_ALINDI, "Ses klonu için hak/izin onayı gerekli."
assert XTTS_LISANSINI_KABUL_EDIYORUM, "XTTS-v2 lisansını (ticari olmayan) kabul etmelisin."

In [ ]:
# ===== İSTEĞE BAĞLI ÇEVİRİ ANAHTARI (gizli) — yoksa yerel ücretsiz çeviri =====
# Kaggle: Add-ons → Secrets → DUB_LLM_API_KEY   |   Colab: soldaki 🔑 → DUB_LLM_API_KEY
import os
key = None
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("DUB_LLM_API_KEY")
except Exception:
    try:
        from google.colab import userdata
        key = userdata.get("DUB_LLM_API_KEY")
    except Exception:
        pass
if key:
    os.environ["DUB_LLM_API_KEY"] = key
    os.environ["DUB_LLM_BASE_URL"] = CEVIRI_BASE_URL
    os.environ["DUB_LLM_MODEL"] = CEVIRI_MODEL
    print("Çeviri: API (anahtar ayarlandı).")
else:
    os.environ.pop("DUB_LLM_API_KEY", None)
    print("Çeviri: yerel NLLB-200 (ücretsiz).")

In [ ]:
# ===== KURULUM =====
import os, subprocess
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(BASE)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu-core libgl1 > /dev/null
!rm -rf {BASE}/VIDEO-OTOMASYON && git clone -q --depth 1 -b {REPO_BRANCH} {REPO} {BASE}/VIDEO-OTOMASYON
!pip install -q coqui-tts "demucs==4.0.1" "faster-whisper==1.1.1" "edge-tts==7.0.0" sentencepiece requests

In [ ]:
# ===== DUDAK SENKRONU KURULUMU (ayrı ortam; yalnızca DUDAK_SENKRONU=True ise) =====
LS_DIR = f"{BASE}/LatentSync"
LS_PY = f"{BASE}/ls-venv/bin/python"
if DUDAK_SENKRONU:
    !pip install -q virtualenv && virtualenv -q {BASE}/ls-venv
    ![ -d {LS_DIR} ] || git clone -q --depth 1 https://github.com/bytedance/LatentSync {LS_DIR}
    !cd {LS_DIR} && {BASE}/ls-venv/bin/pip install -q -r requirements.txt "huggingface_hub[cli]"
    # 1.5 modeli T4'ün 16 GB belleğine sığar (1.6 ≈ 18 GB ister)
    !cd {LS_DIR} && {BASE}/ls-venv/bin/huggingface-cli download ByteDance/LatentSync-1.5 latentsync_unet.pt whisper/tiny.pt --local-dir checkpoints

In [ ]:
# ===== DUBLAJ =====
import subprocess, os
env = dict(os.environ, COQUI_TOS_AGREED="1", DUB_OUTPUT_DIR=f"{BASE}/output")
cmd = ["python", "dub_clone.py", VIDEO, "--izinli", "--whisper-model", WHISPER_MODEL]
if KAYNAK_DIL: cmd += ["--source-lang", KAYNAK_DIL]
if ALTYAZI_GOM: cmd += ["--burn-subtitles"]
if DUDAK_SENKRONU: cmd += ["--lipsync", "--latentsync-dir", LS_DIR, "--latentsync-python", LS_PY]
p = subprocess.Popen(cmd, cwd=f"{BASE}/VIDEO-OTOMASYON", env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
assert p.wait() == 0, "Dublaj başarısız (yukarıdaki hataya bak)."

In [ ]:
# ===== SONUÇ =====
from IPython.display import Video, display
out = f"{BASE}/output/video_dublaj.mp4"
display(Video(out, embed=True, width=480))
try:
    from google.colab import files
    files.download(out); files.download(f"{BASE}/output/dublaj_tr.srt")
except Exception:
    print("Kaggle: dosyalar sağdaki Output bölümünde →", f"{BASE}/output/")